# Text Preprocessing and Feature Extraction

## Objective

This notebook prepares the cleaned energy app review text for machine learning.

The goal is to convert text into numerical features using:

1. CountVectorizer
2. TF-IDF Vectorizer

These feature matrices will later be used to train Naive Bayes and Logistic Regression models for sentiment classification and rating prediction.

In [1]:
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
data_path = "../data/processed/energy_app_reviews_cleaned.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (3070, 14)


,app_name,app_id,category,country,review_text,rating,review_date,thumbs_up_count,app_version,word_count,char_count,sentiment,clean_review_text,clean_word_count
0,Octopus Electroverse,energy.octopus.electricjuice.android,EV Charging / Energy Mobility,gb,"Good app, but why does the map always default to Charing Cross? It would be better if there were choices - default/h...",4,2026-05-30 09:56:07,0,2026.05.12,24,133,Positive,good app map always default charing cross would better choice default home last used,14
1,Octopus Electroverse,energy.octopus.electricjuice.android,EV Charging / Energy Mobility,gb,fantastic,5,2026-05-29 03:06:33,0,2026.05.05,1,9,Positive,fantastic,1
2,Octopus Electroverse,energy.octopus.electricjuice.android,EV Charging / Energy Mobility,gb,I'll be honest and say when I decided to go for an Electric Car I was very nervous regarding charging and the networ...,5,2026-05-27 18:31:56,0,2026.05.05,47,259,Positive,honest say decided go electric car nervous regarding charging network support however installing app game changer re...,22
3,Octopus Electroverse,energy.octopus.electricjuice.android,EV Charging / Energy Mobility,gb,"The app seems fine, and the support is excellent! I had a problem with my account, not being able to log in, and I c...",5,2026-05-27 10:23:39,0,2026.05.05,45,212,Positive,app seems fine support excellent problem account able log could reply email answer human sorted thanks ryan l,18
4,Octopus Electroverse,energy.octopus.electricjuice.android,EV Charging / Energy Mobility,gb,Very easy and fast to use also it covers a wide range of charge points,5,2026-05-27 05:55:17,0,2026.05.05,15,70,Positive,easy fast use also cover wide range charge point,9


In [3]:
text_column = "clean_review_text"
sentiment_target = "sentiment"
rating_target = "rating"

X = df[text_column]
y_sentiment = df[sentiment_target]
y_rating = df[rating_target]

print("Number of reviews:", len(X))
print("\nSentiment distribution:")
print(y_sentiment.value_counts())

print("\nRating distribution:")
print(y_rating.value_counts().sort_index())

Number of reviews: 3070

Sentiment distribution:
sentiment
Negative    1439
Positive    1411
Neutral      220
Name: count, dtype: int64

Rating distribution:
rating
1    1229
2     210
3     220
4     234
5    1177
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train_sentiment, y_test_sentiment = train_test_split(
    X,
    y_sentiment,
    test_size=0.2,
    random_state=42,
    stratify=y_sentiment
)

print("Training reviews:", X_train.shape[0])
print("Testing reviews:", X_test.shape[0])

print("\nTraining sentiment distribution:")
print(y_train_sentiment.value_counts())

print("\nTesting sentiment distribution:")
print(y_test_sentiment.value_counts())

Training reviews: 2456
Testing reviews: 614

Training sentiment distribution:
sentiment
Negative    1151
Positive    1129
Neutral      176
Name: count, dtype: int64

Testing sentiment distribution:
sentiment
Negative    288
Positive    282
Neutral      44
Name: count, dtype: int64


In [5]:
custom_stopwords = [
    "app",
    "apps",
    "nu",
    "sa",
    "la",
    "si"
]

custom_stopwords

['app', 'apps', 'nu', 'sa', 'la', 'si']

In [6]:
count_vectorizer = CountVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    stop_words=custom_stopwords,
    token_pattern=r"(?u)\b[a-zA-Z]{2,}\b"
)

X_train_count = count_vectorizer.fit_transform(X_train)
X_test_count = count_vectorizer.transform(X_test)

print("CountVectorizer training matrix shape:", X_train_count.shape)
print("CountVectorizer testing matrix shape:", X_test_count.shape)

CountVectorizer training matrix shape: (2456, 2318)
CountVectorizer testing matrix shape: (614, 2318)


In [7]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    stop_words=custom_stopwords,
    token_pattern=r"(?u)\b[a-zA-Z]{2,}\b"
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF training matrix shape:", X_train_tfidf.shape)
print("TF-IDF testing matrix shape:", X_test_tfidf.shape)

TF-IDF training matrix shape: (2456, 2318)
TF-IDF testing matrix shape: (614, 2318)


In [8]:
feature_summary = pd.DataFrame({
    "feature_method": ["CountVectorizer", "TF-IDF"],
    "train_rows": [X_train_count.shape[0], X_train_tfidf.shape[0]],
    "test_rows": [X_test_count.shape[0], X_test_tfidf.shape[0]],
    "number_of_features": [X_train_count.shape[1], X_train_tfidf.shape[1]],
    "ngram_range": ["1 to 2", "1 to 2"],
    "max_features": [5000, 5000],
    "min_df": [3, 3],
    "max_df": [0.90, 0.90]
})

feature_summary

,feature_method,train_rows,test_rows,number_of_features,ngram_range,max_features,min_df,max_df
0,CountVectorizer,2456,614,2318,1 to 2,5000,3,0.9
1,TF-IDF,2456,614,2318,1 to 2,5000,3,0.9


In [9]:
feature_summary.to_csv("../reports/feature_extraction_summary.csv", index=False)

print("Feature extraction summary saved to ../reports/feature_extraction_summary.csv")

Feature extraction summary saved to ../reports/feature_extraction_summary.csv


In [10]:
def get_top_features(vectorizer, feature_matrix, top_n=25):
    """
    Return the most frequent / highest-weighted features from a vectorized text matrix.
    """

    feature_names = vectorizer.get_feature_names_out()
    feature_scores = feature_matrix.sum(axis=0).A1

    features_df = pd.DataFrame({
        "feature": feature_names,
        "score": feature_scores
    })

    return features_df.sort_values("score", ascending=False).head(top_n)

In [11]:
top_count_features = get_top_features(
    count_vectorizer,
    X_train_count,
    top_n=25
)

top_count_features

,feature,score
2256,work,350
2134,use,336
303,charging,302
1984,time,264
291,charger,257
577,easy,242
841,good,230
2109,update,225
866,great,212
464,de,210


In [12]:
top_tfidf_features = get_top_features(
    tfidf_vectorizer,
    X_train_tfidf,
    top_n=25
)

top_tfidf_features

,feature,score
841,good,79.927137
577,easy,71.203834
866,great,66.919967
2134,use,64.024444
2256,work,61.052356
303,charging,45.767959
291,charger,41.803337
589,easy use,41.507961
1984,time,39.872150
2109,update,34.981873


In [13]:
top_count_features.to_csv("../reports/top_countvectorizer_features.csv", index=False)
top_tfidf_features.to_csv("../reports/top_tfidf_features.csv", index=False)

print("Top feature files saved successfully.")

Top feature files saved successfully.


## Feature Extraction Observations

CountVectorizer represents review text using raw word and phrase counts. This is simple and works well as a baseline, especially with Naive Bayes.

TF-IDF represents text by giving more importance to terms that are frequent in a review but not too common across all reviews. This often works better with Logistic Regression.

The use of unigrams and bigrams helps capture both individual words and short phrases such as "not working", "customer service", "charging station", and "meter reading".

The extracted features will be used in the next step to train and compare machine learning models.